<a href="https://colab.research.google.com/github/tymepas/GenAI_30days/blob/main/GAI_D8_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple RNN

In [1]:
import numpy as np

In [2]:
class SimpleRNN:
  def __init__(self, input_size, hidden_size, output_size):
     self.hidden_size = hidden_size
     self.Wx = np.random.randn(input_size, hidden_size)
     self.Wh = np.random.randn(hidden_size, hidden_size)
     self.Wy = np.random.randn(hidden_size, output_size)
     self.bh = np.zeros((1, hidden_size))
     self.by = np.zeros((1, output_size))

  def forward(self, inputs):
    h = np.zeros((inputs.shape[0], self.hidden_size))
    outputs = []

    for t in range(inputs.shape[1]):
      h = np.tanh(np.dot(inputs[:, t, :], self.Wx) + np.dot(h, self.Wh) + self.bh)
      y = np.dot(h, self.Wy) + self.by
      outputs.append(y)
    return np.array(outputs), h

inputs = np.random.randn(5, 10, 8)
rnn = SimpleRNN(input_size=8, hidden_size=16, output_size=4)

outputs, h = rnn.forward(inputs)

print("Outputs shape:", outputs.shape)
print("Final hidden state:", h)

Outputs shape: (10, 5, 4)
Final hidden state: [[-0.99998311 -0.67483064 -0.99745377 -0.68671962  0.99999999  0.98461218
  -0.34078048  0.97428097  0.99955859 -0.0479266  -0.93839206  0.99999928
  -0.99999991  0.9896115   0.92243094  0.99999913]
 [-0.99998371 -0.99965432 -1.         -0.99999999  0.99561525 -0.99998561
  -0.67437592  0.97985583 -0.99999463 -0.99243705  0.99999998  0.99959473
   0.99999609 -0.99996092 -0.99222805  0.96899255]
 [ 0.77758383  0.99990986 -0.99999593 -0.99999843  0.99985656  0.90524359
   0.99932851  0.99999461 -0.99497923  0.99288947 -0.99952982  0.99466208
   0.9010377   0.98019943  0.99923814  0.99998339]
 [-0.99971599  0.35731414  0.99998961  0.99986983  0.99697723  0.9999294
   0.99944507 -0.94065896 -0.99944914  0.9971224   0.37864003  0.99998535
  -0.99994778  0.99998694 -0.99298071  1.        ]
 [-0.9995112  -0.08291061  0.94040547  0.99999996 -0.70441459 -0.99998627
   0.99997781  0.99137935  0.91895875  0.93154759 -0.61040958 -0.90599249
  -0.970156

Hands-on


# Hands-on

In [3]:
import tensorflow as tf
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding
from tensorflow.keras.models import Sequential
import numpy as np

In [4]:
url = 'https://www.gutenberg.org/files/100/100-0.txt'
text_path = tf.keras.utils.get_file('shakespeare.txt', url)

5422721/5422721 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [5]:
text = open(text_path, 'rb').read().decode(encoding='utf-8')

In [6]:
vocab = sorted(set(text))
char2idx = {char: idx for idx, char in enumerate(vocab)}
idx2char = np.array(vocab)

In [7]:
text_as_int = np.array([char2idx[c] for c in text])

In [8]:
seq_length = 100
examples_per_epoch = len(text) // seq_length

In [9]:
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

In [10]:
def split_input_target(chunk):
  input_text = chunk[:-1]
  target_text = chunk[1:]
  return input_text, target_text

dataset = sequences.map(split_input_target)

In [11]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

In [12]:
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 1024

In [13]:
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=seq_length),
    SimpleRNN(rnn_units, return_sequences=True),
    Dense(vocab_size)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [14]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

In [15]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
EPOCHS = 10
history = model.fit(dataset, epochs=EPOCHS)

Epoch 1/10
829/829 ━━━━━━━━━━━━━━━━━━━━ 45s 47ms/step - loss: 4.5589
Epoch 2/10
829/829 ━━━━━━━━━━━━━━━━━━━━ 41s 47ms/step - loss: 4.4263
Epoch 3/10
829/829 ━━━━━━━━━━━━━━━━━━━━ 42s 48ms/step - loss: 4.3311
Epoch 4/10
829/829 ━━━━━━━━━━━━━━━━━━━━ 42s 48ms/step - loss: 5.7416
Epoch 5/10
829/829 ━━━━━━━━━━━━━━━━━━━━ 41s 48ms/step - loss: 5.4549
Epoch 6/10
829/829 ━━━━━━━━━━━━━━━━━━━━ 82s 48ms/step - loss: 4.5843
Epoch 7/10
829/829 ━━━━━━━━━━━━━━━━━━━━ 42s 48ms/step - loss: 4.3245
Epoch 8/10
829/829 ━━━━━━━━━━━━━━━━━━━━ 41s 47ms/step - loss: 4.5750
Epoch 9/10
829/829 ━━━━━━━━━━━━━━━━━━━━ 41s 47ms/step - loss: 5.6512
Epoch 10/10
829/829 ━━━━━━━━━━━━━━━━━━━━ 41s 47ms/step - loss: 4.8950


In [18]:
def generate_text(model, start_string):
    num_generate = 1000
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)  # ✅ ADD THIS LINE — converts list to tensor
    text_generated = []

    temperature = 1.0

    for layer in model.layers:
        if hasattr(layer, 'reset_states'):
            layer.reset_states()

    for _ in range(num_generate):
        predictions = model(input_eval)           # ✅ Now receives a proper tensor
        predictions = tf.squeeze(predictions, 0)

        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()

        input_eval = tf.expand_dims([predicted_id], 0)  # This line was already correct
        text_generated.append(idx2char[predicted_id])

    return start_string + ''.join(text_generated)


print(generate_text(model, start_string="ROMEO: "))

ROMEO: CKEISC
KEKEmik,rÇKEUSSKERKEKEC
KEKECSSKERKUKEakniktZS kyzep)ikvik)oikr kukk,0kkkkikkniksXksx,N-djukp’k7Whe,
KEvikfhZUKTIKEiknikvjw
KE’k, ke,) t, k kp051kkukt kkkkllf0kk’kikv
KEnike,ÇKECICDupm_
KE
KEnik kniklik, fikpik	rakuknikd, kikukæw
KEmavik,
KEZpikkuur
KEKE
KEK0kkkakrak…niktâkte,J
KESS._KESc,
KERKEOKDov
KEmKERKECSKECKEKE
KEnikyw
KECKESSKESPSEplp,s;_UUKEik…oikufikes’d k fikvikhe,ERKEikuk,
KEP
KELRKE
KEKEanikeikrik'‘3mafik,Z2QE’sikrik,?
KEK:
KECKquukke,5pken t krKECKECKEKEc’s.
KEmikr k’nt*kiknikkunikp k,s’kg
KEnp’tl k9
KEikkna,ÀECKEhiksskvikmed kik,,
KEanik kkkJF
KELSRKESKEKEKEKEfikd,keè1ZEKEnikskrp70HOKERKESKE
KL&krakeikwTICKEP	KEnikuv)32
KESKE
KERKELURKERKEmëphukkksœ kikkumtoavikkuk0kiknaeikla'ikg
KEKEKEW’t…ECACCKEICSKEcikikukuku f73À k k’ikreik1kulikeshuuupr,
KECSSHoiktxrakkrikk, kEjeik,drakikikik,çæ9œvpm?,ZUFKEpe
KEm4
K’tyopik0ut
KERKEKEKECSikikræf d.
KECKESKEv
KEckuk
KE
KEplfeikeaiksssë keikk tlç…ECRK’ mmak,Éikukuniks ksdfà
KEbikkk,
KEC.
K’dockuik kur,t8kky3p:
KEKERKEKEKE